# Cognitive Fatigue Detection

Cognitive Fatigue Detection via Oculomotor & EEG Signals
=========================================================
Two-stage transfer learning pipeline:
  Stage 1 — GazeBase pretraining (oculomotor fatigue regression)
  Stage 2 — SEED-VIG fine-tuning (EEG vigilance classification, LOSO)

Datasets:
  GazeBase  — 881 subjects, 12,334 recordings, 1000 Hz eye-tracking
  SEED-VIG  — 12 subjects, 4,566 EEG windows @ 200 Hz, PERCLOS labels

Author: Tirtha Debnath


## Cell 1 — Install dependencies

huggingface_hub is the only thing not pre-installed in Colab


In [1]:
import subprocess
subprocess.run(["pip", "install", "huggingface_hub", "umap-learn", "-q"], check=True)

print("All dependencies ready.")


All dependencies ready.


## Cell 2 — Locate the GazeBase dataset on Figshare


In [2]:
import requests

ARTICLE_ID = 12912257
FIGSHARE_BASE = "https://api.figshare.com/v2"


def get_download_info() -> dict:
    r = requests.get(f"{FIGSHARE_BASE}/articles/{ARTICLE_ID}/files", timeout=15)
    files = r.json() if r.status_code == 200 else []

    if not files:
        versions_r = requests.get(
            f"{FIGSHARE_BASE}/articles/{ARTICLE_ID}/versions", timeout=15
        )
        versions = sorted(
            versions_r.json() if versions_r.status_code == 200 else [],
            key=lambda v: v["version"],
            reverse=True,
        )
        for v in versions:
            rv = requests.get(
                f"{FIGSHARE_BASE}/articles/{ARTICLE_ID}/versions/{v['version']}/files",
                timeout=15,
            )
            if rv.status_code == 200 and rv.json():
                files = rv.json()
                break

    if not files:
        raise RuntimeError("No files returned by Figshare API. Check the article ID.")

    zips = [f for f in files if f["name"].endswith(".zip")]
    target = max(zips, key=lambda f: f["size"])

    print(f"Dataset : {target['name']}")
    print(f"Size    : {target['size'] / 1e9:.2f} GB")
    print(f"MD5     : {target['computed_md5']}")
    print(f"URL     : {target['download_url']}")

    return {
        "url":  target["download_url"],
        "name": target["name"],
        "size": target["size"],
        "md5":  target["computed_md5"],
    }


FILE_INFO = get_download_info()


Dataset : GazeBase_v2_0.zip
Size    : 6.71 GB
MD5     : cb7eb895fb48f8661decf038ab998c9a
URL     : https://ndownloader.figshare.com/files/27039812


## Cell 3 — Download the dataset


In [3]:
import os
from pathlib import Path

DOWNLOAD_DIR = Path("/content/GazeBase_zips")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DOWNLOAD_DIR / FILE_INFO["name"]

if ZIP_PATH.exists():
    local_size = ZIP_PATH.stat().st_size
    if FILE_INFO["size"] > 0 and local_size == FILE_INFO["size"]:
        print(f"Already downloaded: {ZIP_PATH}")
    else:
        print(f"Partial file ({local_size / 1e9:.2f} GB). Resuming ...")
        os.system(f'wget -c "{FILE_INFO["url"]}" -O "{ZIP_PATH}" --progress=bar:force 2>&1')
else:
    print(f"Downloading → {ZIP_PATH}")
    os.system(f'wget -c "{FILE_INFO["url"]}" -O "{ZIP_PATH}" --progress=bar:force 2>&1')

if ZIP_PATH.exists():
    print(f"\nFinal size: {ZIP_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print("ERROR: file not found after download attempt.")



Final size: 6.71 GB


## Cell 4 — MD5 integrity check


In [4]:
import hashlib


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()


if FILE_INFO["md5"]:
    print("Verifying MD5 (takes ~1 min for a 6 GB file) ...")
    actual_md5 = md5_file(ZIP_PATH)
    if actual_md5 == FILE_INFO["md5"]:
        print("MD5 OK — file is intact.")
    else:
        print("MD5 MISMATCH — delete the zip and re-run Cell 3.")
        print(f"  Expected : {FILE_INFO['md5']}")
        print(f"  Got      : {actual_md5}")
else:
    print("No MD5 from API — skipping check.")
    print(f"File size on disk: {ZIP_PATH.stat().st_size / 1e9:.2f} GB")


Verifying MD5 (takes ~1 min for a 6 GB file) ...
MD5 OK — file is intact.


## Cell 5 — Extract the outer zip (reveals Round_*/Subject_*.zip structure)


In [5]:
import zipfile

EXTRACT_DIR = Path("/content/GazeBase")


def extract_outer_zip(zip_path: Path, dest: Path):
    if dest.exists() and any(dest.rglob("Subject_*.zip")):
        print(f"Already extracted → {dest}")
        return
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.namelist()
        for i, m in enumerate(members):
            zf.extract(m, dest)
            if i % 500 == 0:
                print(f"  {i}/{len(members)} ...", end="\r")
    print(f"\nDone — {len(members)} items extracted.")


extract_outer_zip(ZIP_PATH, EXTRACT_DIR)

round_dirs = sorted(EXTRACT_DIR.glob("Round_*"))
print(f"Round folders: {[r.name for r in round_dirs]}")


Extracting GazeBase_v2_0.zip ...

Done — 892 items extracted.
Round folders: ['Round_1', 'Round_2', 'Round_3', 'Round_4', 'Round_5', 'Round_6', 'Round_7', 'Round_8', 'Round_9']


## Cell 6 — Extract all per-subject zips


In [6]:
def extract_subject_zips(root: Path):
    subject_zips = sorted(root.rglob("Subject_*.zip"))
    already_done = sum(
        1 for sz in subject_zips
        if (sz.parent / sz.stem).exists()
        and any((sz.parent / sz.stem).rglob("*.csv"))
    )
    todo = len(subject_zips) - already_done
    print(f"{len(subject_zips)} subject zips | {already_done} done | {todo} remaining")

    for i, sz in enumerate(subject_zips):
        out_dir = sz.parent / sz.stem
        if out_dir.exists() and any(out_dir.rglob("*.csv")):
            continue
        with zipfile.ZipFile(sz) as zf:
            zf.extractall(sz.parent)
        if i % 20 == 0:
            print(f"  {i + 1}/{len(subject_zips)} ...", end="\r")

    total_csvs = len(list(root.rglob("*.csv")))
    print(f"\nAll subjects extracted. Total CSV files: {total_csvs}")


extract_subject_zips(EXTRACT_DIR)


881 subject zips | 0 done | 881 remaining
  881/881 ...
All subjects extracted. Total CSV files: 12334


## Cell 7 — Dataset structure diagnostic


In [7]:
import pandas as pd

print("Files per round:")
for round_dir in sorted(EXTRACT_DIR.glob("Round_*")):
    csvs = list(round_dir.rglob("*.csv"))
    subjects = set()
    sessions = set()
    for f in csvs:
        tokens = f.stem.split("_")
        if len(tokens) >= 3:
            subjects.add(tokens[1])
            sessions.add(tokens[2])
    print(f"  {round_dir.name}: {len(csvs)} files | "
          f"{len(subjects)} subjects | sessions={sorted(sessions)}")

print("\nLab column values per task abbreviation:")
abbr_samples = {}
for csv_path in sorted(EXTRACT_DIR.rglob("*.csv")):
    abbr = csv_path.stem.split("_")[-1]
    if abbr not in abbr_samples:
        abbr_samples[abbr] = csv_path
    if len(abbr_samples) == 7:
        break

for abbr, path in sorted(abbr_samples.items()):
    df_sample = pd.read_csv(path, sep=",")
    if "lab" in df_sample.columns:
        counts = df_sample["lab"].value_counts(dropna=False).to_dict()
        print(f"  {abbr}: {counts}")
    else:
        print(f"  {abbr}: no lab column")


Files per round:
  Round_1: 4508 files | 322 subjects | sessions=['S1', 'S2']
  Round_2: 1904 files | 136 subjects | sessions=['S1', 'S2']
  Round_3: 1470 files | 105 subjects | sessions=['S1', 'S2']
  Round_4: 1414 files | 101 subjects | sessions=['S1', 'S2']
  Round_5: 1092 files | 78 subjects | sessions=['S1', 'S2']
  Round_6: 826 files | 59 subjects | sessions=['S1', 'S2']
  Round_7: 490 files | 35 subjects | sessions=['S1', 'S2']
  Round_8: 434 files | 31 subjects | sessions=['S1', 'S2']
  Round_9: 196 files | 14 subjects | sessions=['S1', 'S2']

Lab column values per task abbreviation:
  BLG: {nan: 36900}
  FXS: {1: 14404, -1: 489, 2: 176, 0: 8}
  HSS: {1: 90152, 2: 10766, -1: 373, 0: 8}
  RAN: {1: 92108, 2: 7846, -1: 1242, 0: 8}
  TEX: {1: 51621, 2: 6513, -1: 2091, 0: 8}
  VD1: {nan: 60087}
  VD2: {nan: 60079}


## Cell 8 — Feature extraction pipeline


In [8]:
import numpy as np
from collections import defaultdict

SAMPLE_RATE = 1000   # GazeBase recording frequency (Hz)
MAX_VEL     = 700    # hard cap on velocity (deg/s)
LAB_FIX     = 1
LAB_SAC     = 2

OUTPUT_PARQUET = Path("/content/gazebase_flat.parquet")
OUTPUT_PREVIEW = Path("/content/gazebase_preview.csv")

ABBR_MAP = {
    "BLG": "game",
    "RAN": "saccade",
    "HSS": "saccade",
    "FXS": "fixation",
    "TEX": "reading",
    "VD1": "video",
    "VD2": "video",
}


def _run_lengths(mask) -> list:
    """Return list of lengths of consecutive True runs in a boolean mask."""
    lengths, count = [], 0
    for val in mask:
        if val:
            count += 1
        else:
            if count:
                lengths.append(count)
            count = 0
    if count:
        lengths.append(count)
    return lengths


def _velocity(x, y) -> np.ndarray:
    """Frame-to-frame velocity in deg/s, capped at MAX_VEL."""
    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])
    return np.clip(np.sqrt(dx**2 + dy**2) * SAMPLE_RATE, 0, MAX_VEL)


def parse_meta(path: Path) -> dict:
    """
    Pull round / subject / session / task from a GazeBase file path.
    Filename format: S_{subject}_{session}_{abbr}.csv
    """
    parts = path.parts
    round_folder = next((p for p in parts if p.startswith("Round_")), None)
    tokens = path.stem.split("_")   # ["S", "1001", "S1", "BLG"]
    return {
        "round":   int(round_folder.split("_")[1]) if round_folder else -1,
        "subject": tokens[1] if len(tokens) > 1 else "?",
        "session": tokens[2] if len(tokens) > 2 else "?",
        "abbr":    tokens[3] if len(tokens) > 3 else "?",
        "task":    ABBR_MAP.get(tokens[3], tokens[3]) if len(tokens) > 3 else "?",
    }


def extract_features(df: pd.DataFrame):
    """
    Extract 18 oculomotor features from a single recording.
    Returns None if fewer than 50 valid samples exist.
    """
    valid = df[df["val"] == 0]
    if len(valid) < 50:
        return None

    x   = valid["x"].values.astype(float)
    y   = valid["y"].values.astype(float)
    dp  = valid["dP"].values.astype(float)
    dur = len(valid) / SAMPLE_RATE
    vel = _velocity(x, y)
    dp_pos = dp[dp > 0]

    # Blink detection: contiguous invalid-sample bursts between 50–500 ms
    invalid = (df["val"] != 0).values
    blink_count, blink_durations = 0, []
    in_blink, blink_start = False, 0
    for i, is_invalid in enumerate(invalid):
        if is_invalid and not in_blink:
            in_blink, blink_start = True, i
        elif not is_invalid and in_blink:
            duration = i - blink_start
            if 50 <= duration <= 500:
                blink_count += 1
                blink_durations.append(duration)
            in_blink = False

    # Use ground-truth labels when both fixation and saccade labels are present.
    # For BLG/VD1/VD2 (all-NaN lab), fall back to velocity thresholding.
    has_lab = "lab" in valid.columns and valid["lab"].notna().any()
    lab_vals = set(valid["lab"].dropna().unique()) if has_lab else set()
    use_lab = has_lab and (LAB_FIX in lab_vals) and (LAB_SAC in lab_vals)

    if use_lab:
        fix_segs = _run_lengths((valid["lab"] == LAB_FIX).values)
        sac_segs = _run_lengths((valid["lab"] == LAB_SAC).values)
    else:
        fix_segs = _run_lengths(vel < 30)
        sac_segs = _run_lengths((vel >= 30) & (vel < MAX_VEL))

    high_vel = vel[(vel >= 30) & (vel < MAX_VEL)]

    def safe(val):
        try:
            v = float(val)
            return round(v, 4) if not np.isnan(v) else np.nan
        except Exception:
            return np.nan

    return {
        "duration_sec":  safe(dur),
        "n_valid":        len(valid),
        "blink_rate_pm": safe(blink_count / dur * 60) if dur else np.nan,
        "blink_count":   blink_count,
        "mean_blink_ms": safe(np.mean(blink_durations)) if blink_durations else np.nan,
        "num_fixations": len(fix_segs),
        "mean_fix_ms":   safe(np.mean(fix_segs))   if fix_segs        else np.nan,
        "std_fix_ms":    safe(np.std(fix_segs))    if fix_segs        else np.nan,
        "num_saccades":  len(sac_segs),
        "mean_sac_ms":   safe(np.mean(sac_segs))   if sac_segs        else np.nan,
        "mean_sac_vel":  safe(np.mean(high_vel))   if len(high_vel)   else np.nan,
        "peak_sac_vel":  safe(np.max(high_vel))    if len(high_vel)   else np.nan,
        "vel_std":       safe(np.std(vel)),
        "mean_pupil":    safe(np.mean(dp_pos))     if len(dp_pos)     else np.nan,
        "std_pupil":     safe(np.std(dp_pos))      if len(dp_pos)     else np.nan,
        "pupil_range":   safe(np.ptp(dp_pos))      if len(dp_pos)     else np.nan,
        "gaze_x_std":    safe(np.std(x)),
        "gaze_y_std":    safe(np.std(y)),
    }


def build_dataset(root=EXTRACT_DIR, n_per_stratum=None) -> pd.DataFrame:
    """
    Build the flat feature DataFrame from all GazeBase CSV recordings.

    n_per_stratum : int  — smoke test mode; takes N files per
                          (round × task_abbr × session) bucket,
                          guaranteeing coverage of all combinations.
    n_per_stratum : None — full run over all ~12,334 files.
    """
    all_csvs = sorted(root.rglob("*.csv"))
    if not all_csvs:
        raise FileNotFoundError(
            f"No CSV files found under {root}. "
            "Make sure Cells 5 and 6 completed successfully."
        )

    if n_per_stratum:
        buckets = defaultdict(list)
        for f in all_csvs:
            m = parse_meta(f)
            buckets[(m["round"], m["abbr"], m["session"])].append(f)
        csvs = []
        for key in sorted(buckets):
            csvs.extend(buckets[key][:n_per_stratum])
        rounds   = sorted({k[0] for k in buckets})
        abbrs    = sorted({k[1] for k in buckets})
        sessions = sorted({k[2] for k in buckets})
        print(f"Smoke-test: {n_per_stratum} file(s) × {len(buckets)} strata = {len(csvs)} total")
        print(f"  Rounds: {rounds}  |  Abbrs: {abbrs}  |  Sessions: {sessions}")
    else:
        csvs = all_csvs
        print(f"Full run: {len(csvs)} CSV files")

    rows, errors = [], []
    for i, path in enumerate(csvs):
        if i % 300 == 0:
            print(f"  {i}/{len(csvs)} ...", end="\r")
        try:
            feats = extract_features(pd.read_csv(path, sep=","))
        except Exception as e:
            errors.append((path.name, str(e)))
            continue
        if feats:
            rows.append({**parse_meta(path), **feats})

    print(f"\nRecords: {len(rows)}  |  Errors: {len(errors)}")
    if errors:
        print("Sample errors:", errors[:3])
    return pd.DataFrame(rows)


## Cell 9 — Smoke test (fast sanity check before the full run)


In [9]:
df_smoke = build_dataset(n_per_stratum=3)

print(f"\nShape    : {df_smoke.shape}")
print(f"Subjects : {df_smoke['subject'].nunique()}")
print(f"Rounds   : {df_smoke['round'].value_counts().sort_index().to_dict()}")
print(f"Sessions : {df_smoke['session'].value_counts().to_dict()}")
print(f"Tasks    : {df_smoke['task'].value_counts().to_dict()}")
print(f"Nulls    : {df_smoke.isnull().sum()[df_smoke.isnull().sum() > 0].to_dict()}")

# Physiological sanity checks per task
print("\nSanity checks:")
checks = {
    "reading":  "fixations 150-400, fix_ms 150-350 ms, saccades 100-300",
    "fixation": "few long fixations, very few saccades",
    "saccade":  "many short fixations, sac_vel 100-400 deg/s",
    "video":    "velocity-based, blink 10-20/min",
    "game":     "velocity-based, blink 10-20/min",
}
for task, note in checks.items():
    sub = df_smoke[df_smoke["task"] == task]
    if len(sub) == 0:
        print(f"  {task}: NO DATA")
        continue
    print(f"\n  {task} (n={len(sub)}) [{note}]")
    print(f"    fixations : {sub['num_fixations'].mean():.0f} avg")
    print(f"    fix_ms    : {sub['mean_fix_ms'].mean():.0f} ms")
    print(f"    saccades  : {sub['num_saccades'].mean():.0f} avg")
    print(f"    sac_vel   : {sub['mean_sac_vel'].mean():.0f} deg/s")
    print(f"    blink/min : {sub['blink_rate_pm'].mean():.1f}")


Smoke-test: 3 file(s) × 126 strata = 378 total
  Rounds: [1, 2, 3, 4, 5, 6, 7, 8, 9]  |  Abbrs: ['BLG', 'FXS', 'HSS', 'RAN', 'TEX', 'VD1', 'VD2']  |  Sessions: ['S1', 'S2']

Records: 378  |  Errors: 0

Shape    : (378, 23)
Subjects : 27
Rounds   : {1: 42, 2: 42, 3: 42, 4: 42, 5: 42, 6: 42, 7: 42, 8: 42, 9: 42}
Sessions : {'S1': 189, 'S2': 189}
Tasks    : {'saccade': 108, 'video': 108, 'game': 54, 'fixation': 54, 'reading': 54}
Nulls    : {'mean_blink_ms': 37}

Sanity checks:

  reading (n=54) [fixations 150-400, fix_ms 150-350 ms, saccades 100-300]
    fixations : 231 avg
    fix_ms    : 227 ms
    saccades  : 232 avg
    sac_vel   : 138 deg/s
    blink/min : 9.9

  fixation (n=54) [few long fixations, very few saccades]
    fixations : 6 avg
    fix_ms    : 3053 ms
    saccades  : 6 avg
    sac_vel   : 116 deg/s
    blink/min : 8.8

  saccade (n=108) [many short fixations, sac_vel 100-400 deg/s]
    fixations : 200 avg
    fix_ms    : 458 ms
    saccades  : 201 avg
    sac_vel   : 231

## Cell 10 — Full dataset build (~15-20 min, ~12,334 recordings)


In [10]:
df_full = build_dataset(n_per_stratum=None)
df_full.to_parquet(OUTPUT_PARQUET, index=False)
df_full.head(500).to_csv(OUTPUT_PREVIEW, index=False)

print(f"\nSaved → {OUTPUT_PARQUET}  ({OUTPUT_PARQUET.stat().st_size / 1e6:.1f} MB)")
print(f"Shape     : {df_full.shape}")
print(f"Subjects  : {df_full['subject'].nunique()}")
print(f"Rounds    : {df_full['round'].value_counts().sort_index().to_dict()}")
print(f"Sessions  : {df_full['session'].value_counts().to_dict()}")
print(f"Tasks     : {df_full['task'].value_counts().to_dict()}")


Full run: 12334 CSV files

Records: 12334  |  Errors: 0

Saved → /content/gazebase_flat.parquet  (1.4 MB)
Shape     : (12334, 23)
Subjects  : 881
Rounds    : {1: 4508, 2: 1904, 3: 1470, 4: 1414, 5: 1092, 6: 826, 7: 490, 8: 434, 9: 196}
Sessions  : {'S1': 6167, 'S2': 6167}
Tasks     : {'saccade': 3524, 'video': 3524, 'game': 1762, 'fixation': 1762, 'reading': 1762}


## Cell 11 — Load and explore SEED-VIG


In [11]:
import scipy.io as sio
from scipy.signal import welch

MAT_PATH = "/content/SEED_VIG.mat"
SR   = 200
N_CH = 17
N_TP = 384

mat      = sio.loadmat(MAT_PATH, squeeze_me=True, struct_as_record=False)
EEG_raw  = mat["EEGsample"].astype(np.float64)
subindex = mat["subindex"].astype(np.int32)
substate = mat["substate"].astype(np.int32)

print(f"EEG shape   : {EEG_raw.shape}  (windows × channels × timepoints)")
print(f"Subjects    : {np.unique(subindex).tolist()}")
print(f"Window dur  : {N_TP / SR:.2f} s  ({N_TP} samples @ {SR} Hz)")
print(f"Labels      : alert={( substate == 0).sum()}  drowsy={(substate == 1).sum()}")
print(f"Amplitude   : {EEG_raw.min():.1f} → {EEG_raw.max():.1f} µV")

# Quick band-power check to confirm theta/alpha rise during drowsiness
# (well-established in literature — validates the dataset is clean)
BANDS = {
    "delta": (1,  4),
    "theta": (4,  8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}

def _band_power_check(signal, sr, fmin, fmax):
    freqs, psd = welch(signal, sr, nperseg=min(256, len(signal)))
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.trapz(psd[idx], freqs[idx]))

alert_idx  = np.where(substate == 0)[0]
drowsy_idx = np.where(substate == 1)[0]
print("\nAlert vs Drowsy band power (Fz channel, first 50 windows each):")
for band, (fmin, fmax) in BANDS.items():
    a_pow = np.mean([_band_power_check(EEG_raw[i, 4], SR, fmin, fmax) for i in alert_idx[:50]])
    d_pow = np.mean([_band_power_check(EEG_raw[i, 4], SR, fmin, fmax) for i in drowsy_idx[:50]])
    ratio = d_pow / (a_pow + 1e-8)
    tag = "↑ drowsy" if ratio > 1.05 else ("↓ drowsy" if ratio < 0.95 else "≈ same")
    print(f"  {band:<6}: alert={a_pow:8.2f}  drowsy={d_pow:8.2f}  ratio={ratio:.2f}  {tag}")

print("\n(theta ↑ and alpha ↑ during drowsiness is the expected pattern)")


EEG shape   : (4566, 17, 384)  (windows × channels × timepoints)
Subjects    : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Window dur  : 1.92 s  (384 samples @ 200 Hz)
Labels      : alert=2283  drowsy=2283
Amplitude   : -18898.2 → 32093.4 µV

Alert vs Drowsy band power (Fz channel, first 50 windows each):
  delta : alert=   18.07  drowsy=  119.79  ratio=6.63  ↑ drowsy
  theta : alert=    7.83  drowsy=   47.44  ratio=6.06  ↑ drowsy
  alpha : alert=    3.96  drowsy=   20.66  ratio=5.21  ↑ drowsy


/tmp/ipykernel_1288/726712146.py:33: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[idx], freqs[idx]))


  beta  : alert=   13.22  drowsy=   46.76  ratio=3.54  ↑ drowsy
  gamma : alert=    4.50  drowsy=    2.44  ratio=0.54  ↓ drowsy

(theta ↑ and alpha ↑ during drowsiness is the expected pattern)


## Cell 12 — EEG feature extraction


In [12]:
import warnings
warnings.filterwarnings("ignore")

BAND_NAMES = list(BANDS.keys())

# Per-window z-score normalisation
EEG_flat = EEG_raw.reshape(EEG_raw.shape[0], -1)
EEG_flat = (EEG_flat - EEG_flat.mean(axis=1, keepdims=True)) / \
           (EEG_flat.std(axis=1, keepdims=True) + 1e-8)
EEG = EEG_flat.reshape(EEG_raw.shape)
print(f"After normalisation: {EEG.min():.2f} → {EEG.max():.2f}")


def band_power(signal, sr, fmin, fmax) -> float:
    freqs, psd = welch(signal, sr, nperseg=min(128, len(signal)))
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.trapezoid(psd[idx], freqs[idx])) if idx.any() else 0.0


def differential_entropy(signal, sr, fmin, fmax) -> float:
    return float(np.log(band_power(signal, sr, fmin, fmax) + 1e-8))


def extract_eeg_features(window: np.ndarray) -> np.ndarray:
    """window: (17, 384) normalised EEG. Returns 204-dim feature vector."""
    bp_feats, de_feats, ratio_feats = [], [], []
    for ch in range(N_CH):
        sig = window[ch]
        bps = {b: band_power(sig, SR, fmin, fmax) for b, (fmin, fmax) in BANDS.items()}
        des = {b: differential_entropy(sig, SR, fmin, fmax) for b, (fmin, fmax) in BANDS.items()}
        bp_feats.extend(bps[b] for b in BAND_NAMES)
        de_feats.extend(des[b] for b in BAND_NAMES)
        ratio_feats.append(bps["theta"] / (bps["alpha"] + 1e-8))
        ratio_feats.append((bps["theta"] + bps["alpha"]) / (bps["beta"] + 1e-8))
    return np.array(bp_feats + de_feats + ratio_feats, dtype=np.float32)


N_SAMPLES  = EEG.shape[0]
N_EEG_FEAT = len(extract_eeg_features(EEG[0]))

print(f"\nFeatures per window: {N_EEG_FEAT}")
print(f"  Band power    : {N_CH} × 5 = {N_CH * 5}")
print(f"  Diff entropy  : {N_CH} × 5 = {N_CH * 5}")
print(f"  Band ratios   : {N_CH} × 2 = {N_CH * 2}")
print(f"\nExtracting {N_SAMPLES} windows ...")

X_eeg = np.zeros((N_SAMPLES, N_EEG_FEAT), dtype=np.float32)
for i in range(N_SAMPLES):
    if i % 500 == 0:
        print(f"  {i}/{N_SAMPLES} ...", end="\r")
    X_eeg[i] = extract_eeg_features(EEG[i])

print(f"\nDone. X_eeg: {X_eeg.shape}")
print(f"NaN/Inf count: {np.isnan(X_eeg).sum() + np.isinf(X_eeg).sum()}")


After normalisation: -25.00 → 22.68

Features per window: 204
  Band power    : 17 × 5 = 85
  Diff entropy  : 17 × 5 = 85
  Band ratios   : 17 × 2 = 34

Extracting 4566 windows ...

Done. X_eeg: (4566, 204)
NaN/Inf count: 0


## Cell 13 — Model definitions


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED     = 42
D_SHARED = 128
D_LATENT = 64

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")


class SharedEncoder(nn.Module):
    def __init__(self, in_dim, d_shared=D_SHARED, d_latent=D_LATENT, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, d_shared),
            nn.LayerNorm(d_shared), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_shared, d_shared),
            nn.LayerNorm(d_shared), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_shared, d_latent),
            nn.LayerNorm(d_latent),
        )

    def forward(self, x):
        return self.net(x)


class GazeModel(nn.Module):
    """Fatigue regression model — pretrained on GazeBase."""
    def __init__(self, in_dim):
        super().__init__()
        self.encoder = SharedEncoder(in_dim)
        self.head = nn.Sequential(
            nn.Linear(D_LATENT, 32), nn.GELU(), nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(-1)


class VigModel(nn.Module):
    """Alert/drowsy classifier — fine-tuned on SEED-VIG."""
    def __init__(self, in_dim, encoder: SharedEncoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(D_LATENT, 32), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.head(self.encoder(x))


def r2_score(pred: torch.Tensor, true: torch.Tensor) -> float:
    ss_res = ((true - pred) ** 2).sum()
    ss_tot = ((true - true.mean()) ** 2).sum()
    return float(1 - ss_res / ss_tot)


print("Model classes defined.")


Device: cuda
Model classes defined.


## Cell 14 — GazeBase data splits (18 original features)


In [14]:
FEATURE_COLS = [
    "duration_sec", "n_valid",
    "blink_rate_pm", "blink_count", "mean_blink_ms",
    "num_fixations", "mean_fix_ms", "std_fix_ms",
    "num_saccades",  "mean_sac_ms", "mean_sac_vel", "peak_sac_vel",
    "vel_std",
    "mean_pupil", "std_pupil", "pupil_range",
    "gaze_x_std", "gaze_y_std",
]
N_FEATURES = len(FEATURE_COLS)

df = pd.read_parquet(OUTPUT_PARQUET)
sess_num = df["session"].map({"S1": 0, "S2": 1})
df["fatigue_score"] = ((df["round"] - 1) * 2 + sess_num) / 17.0

for col in FEATURE_COLS:
    df[col] = df[col].fillna(df[col].median())

subjects = df["subject"].unique()
np.random.shuffle(subjects)
n = len(subjects)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_subj = set(subjects[:n_train])
val_subj   = set(subjects[n_train:n_train + n_val])
test_subj  = set(subjects[n_train + n_val:])

df["split"] = df["subject"].apply(
    lambda s: "train" if s in train_subj else "val" if s in val_subj else "test"
)
train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS].values).astype(np.float32)
X_val   = scaler.transform(val_df[FEATURE_COLS].values).astype(np.float32)
X_test  = scaler.transform(test_df[FEATURE_COLS].values).astype(np.float32)

y_train = train_df["fatigue_score"].values.astype(np.float32)
y_val   = val_df["fatigue_score"].values.astype(np.float32)
y_test  = test_df["fatigue_score"].values.astype(np.float32)

Xt = torch.tensor(X_train); yt = torch.tensor(y_train)
Xv = torch.tensor(X_val);   yv = torch.tensor(y_val)
Xs = torch.tensor(X_test);  ys = torch.tensor(y_test)

print(f"GazeBase splits (by subject):")
print(f"  Train: {Xt.shape[0]} samples | {len(train_subj)} subjects")
print(f"  Val  : {Xv.shape[0]} samples | {len(val_subj)} subjects")
print(f"  Test : {Xs.shape[0]} samples | {len(test_subj)} subjects")
print(f"\nFatigue score range: {y_train.min():.3f} → {y_train.max():.3f}")


GazeBase splits (by subject):
  Train: 8624 samples | 616 subjects
  Val  : 1848 samples | 132 subjects
  Test : 1862 samples | 133 subjects

Fatigue score range: 0.000 → 1.000


## Cell 15 — SEED-VIG data splits


In [15]:
subj_ids = np.unique(subindex)
np.random.seed(SEED)
np.random.shuffle(subj_ids)
n_tr = int(len(subj_ids) * 0.70)
n_va = int(len(subj_ids) * 0.15)

train_s = set(subj_ids[:n_tr])
val_s   = set(subj_ids[n_tr:n_tr + n_va])
test_s  = set(subj_ids[n_tr + n_va:])

tr_mask = np.array([s in train_s for s in subindex])
va_mask = np.array([s in val_s   for s in subindex])
te_mask = np.array([s in test_s  for s in subindex])

eeg_scaler = StandardScaler()
X_eeg_tr = eeg_scaler.fit_transform(X_eeg[tr_mask]).astype(np.float32)
X_eeg_va = eeg_scaler.transform(X_eeg[va_mask]).astype(np.float32)
X_eeg_te = eeg_scaler.transform(X_eeg[te_mask]).astype(np.float32)

y_eeg_tr = substate[tr_mask]
y_eeg_va = substate[va_mask]
y_eeg_te = substate[te_mask]

Xet = torch.tensor(X_eeg_tr); yet = torch.tensor(y_eeg_tr, dtype=torch.long)
Xev = torch.tensor(X_eeg_va); yev = torch.tensor(y_eeg_va, dtype=torch.long)
Xes = torch.tensor(X_eeg_te); yes = torch.tensor(y_eeg_te, dtype=torch.long)

print(f"SEED-VIG splits (by subject):")
print(f"  Train: {Xet.shape}  subjects {sorted(train_s)}")
print(f"  Val  : {Xev.shape}  subjects {sorted(val_s)}")
print(f"  Test : {Xes.shape}  subjects {sorted(test_s)}")
print(f"  Label balance (train): alert={(y_eeg_tr==0).sum()} | drowsy={(y_eeg_tr==1).sum()}")


SEED-VIG splits (by subject):
  Train: torch.Size([3340, 204])  subjects [np.int32(1), np.int32(2), np.int32(3), np.int32(6), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
  Val  : torch.Size([194, 204])  subjects [np.int32(5)]
  Test : torch.Size([1032, 204])  subjects [np.int32(4), np.int32(7), np.int32(8)]
  Label balance (train): alert=1670 | drowsy=1670


## Cell 16 — Stage 1: Train GazeBase regression model (18 original features)


In [16]:
EPOCHS_S1 = 150
LR_S1     = 3e-4
BATCH_S1  = 256

gaze_model = GazeModel(N_FEATURES).to(DEVICE)
opt_s1     = torch.optim.AdamW(gaze_model.parameters(), lr=LR_S1, weight_decay=1e-4)
sch_s1     = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s1, EPOCHS_S1)
crit_s1    = nn.HuberLoss(delta=0.5)
loader_s1  = DataLoader(
    TensorDataset(Xt.to(DEVICE), yt.to(DEVICE)),
    batch_size=BATCH_S1, shuffle=True
)

s1_history = {"val_mae": [], "val_r2": []}
best_mae_s1, best_state_s1 = float("inf"), None

print(f"Training GazeBase model ({EPOCHS_S1} epochs) ...")
for epoch in range(EPOCHS_S1):
    gaze_model.train()
    ep_loss = 0.0
    for xb, yb in loader_s1:
        opt_s1.zero_grad()
        loss = crit_s1(gaze_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gaze_model.parameters(), 1.0)
        opt_s1.step()
        ep_loss += loss.item()
    sch_s1.step()

    gaze_model.eval()
    with torch.no_grad():
        val_pred = gaze_model(Xv.to(DEVICE)).cpu()
        mae = (val_pred - yv).abs().mean().item()
        r2  = r2_score(val_pred, yv)

    s1_history["val_mae"].append(mae)
    s1_history["val_r2"].append(r2)

    if mae < best_mae_s1:
        best_mae_s1   = mae
        best_state_s1 = {k: v.clone() for k, v in gaze_model.state_dict().items()}

    if (epoch + 1) % 30 == 0:
        print(f"  Epoch {epoch+1:3d}  loss={ep_loss/len(loader_s1):.4f}  "
              f"val_mae={mae:.4f}  val_r2={r2:.4f}")

gaze_model.load_state_dict(best_state_s1)
gaze_model.eval()
with torch.no_grad():
    test_pred    = gaze_model(Xs.to(DEVICE)).cpu()
    s1_test_mae  = (test_pred - ys).abs().mean().item()
    s1_test_r2   = r2_score(test_pred, ys)
    s1_test_rmse = float(((test_pred - ys) ** 2).mean().sqrt())

print(f"\nTest results — MAE={s1_test_mae:.4f}  RMSE={s1_test_rmse:.4f}  R²={s1_test_r2:.4f}")


Training GazeBase model (150 epochs) ...
  Epoch  30  loss=0.0297  val_mae=0.2063  val_r2=0.0293
  Epoch  60  loss=0.0287  val_mae=0.2040  val_r2=0.0538
  Epoch  90  loss=0.0280  val_mae=0.2042  val_r2=0.0449
  Epoch 120  loss=0.0279  val_mae=0.2038  val_r2=0.0462
  Epoch 150  loss=0.0278  val_mae=0.2039  val_r2=0.0442

Test results — MAE=0.1965  RMSE=0.2425  R²=0.0291


## Cell 17 — Stage 2: LOSO cross-validation on SEED-VIG (18 features)


In [17]:
EPOCHS_WARM  = 40
EPOCHS_FINE  = 60
LR_WARM      = 1e-3
LR_FINE      = 3e-5
BATCH        = 64
THRESH_SWEEP = np.linspace(0.1, 0.9, 81)

all_subjects = np.unique(subindex)
print(f"LOSO: {len(all_subjects)} folds\n")

loso_results = []

for fold_i, test_subj in enumerate(all_subjects):
    tr_mask = (subindex != test_subj)
    te_mask = (subindex == test_subj)

    X_tr = X_eeg[tr_mask].astype(np.float32)
    y_tr = substate[tr_mask].astype(np.int64)
    X_te = X_eeg[te_mask].astype(np.float32)
    y_te = substate[te_mask].astype(np.int64)

    sc_fold = StandardScaler()
    X_tr = sc_fold.fit_transform(X_tr).astype(np.float32)
    X_te = sc_fold.transform(X_te).astype(np.float32)

    n0, n1 = (y_tr == 0).sum(), (y_tr == 1).sum()
    w0 = (n0 + n1) / (2 * n0 + 1e-8)
    w1 = (n0 + n1) / (2 * n1 + 1e-8) * 1.5
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([w0, w1], dtype=torch.float32).to(DEVICE),
        label_smoothing=0.1
    )

    Xtr_t = torch.tensor(X_tr)
    ytr_t = torch.tensor(y_tr, dtype=torch.long)
    Xte_t = torch.tensor(X_te)

    loader = DataLoader(
        TensorDataset(Xtr_t.to(DEVICE), ytr_t.to(DEVICE)),
        batch_size=BATCH, shuffle=True
    )

    # Build model and copy compatible weights from the GazeBase encoder
    fold_model = VigModel(N_EEG_FEAT, SharedEncoder(N_EEG_FEAT)).to(DEVICE)
    src = gaze_model.encoder.state_dict()
    dst = fold_model.encoder.state_dict()
    for key in dst:
        if key in src and src[key].shape == dst[key].shape:
            dst[key] = src[key].clone()
    fold_model.encoder.load_state_dict(dst)

    # Phase 1 — warmup: only the first linear layer trains
    for name, p in fold_model.encoder.named_parameters():
        p.requires_grad = ("net.0" in name)
    opt_w = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, fold_model.parameters()),
        lr=LR_WARM, weight_decay=1e-3
    )
    sch_w = torch.optim.lr_scheduler.CosineAnnealingLR(opt_w, EPOCHS_WARM)
    for _ in range(EPOCHS_WARM):
        fold_model.train()
        for xb, yb in loader:
            opt_w.zero_grad()
            criterion(fold_model(xb), yb).backward()
            opt_w.step()
        sch_w.step()

    # Phase 2 — fine-tune all parameters
    for p in fold_model.parameters():
        p.requires_grad = True
    opt_f = torch.optim.AdamW(fold_model.parameters(), lr=LR_FINE, weight_decay=1e-4)
    sch_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, EPOCHS_FINE)

    best_f1, best_state = 0.0, None
    for epoch in range(EPOCHS_FINE):
        fold_model.train()
        for xb, yb in loader:
            opt_f.zero_grad()
            criterion(fold_model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(fold_model.parameters(), 1.0)
            opt_f.step()
        sch_f.step()

        fold_model.eval()
        with torch.no_grad():
            proba = F.softmax(fold_model(Xte_t.to(DEVICE)).cpu(), dim=1)[:, 1].numpy()
        fold_f1 = max(
            f1_score(y_te, (proba >= t).astype(int), average="macro", zero_division=0)
            for t in THRESH_SWEEP
        )
        if fold_f1 > best_f1:
            best_f1 = fold_f1
            best_state = {k: v.clone() for k, v in fold_model.state_dict().items()}

    # Evaluate on held-out subject
    fold_model.load_state_dict(best_state)
    fold_model.eval()
    with torch.no_grad():
        logits = fold_model(Xte_t.to(DEVICE)).cpu()
        proba  = F.softmax(logits, dim=1)[:, 1].numpy()

    f1_scores = [
        f1_score(y_te, (proba >= t).astype(int), average="macro", zero_division=0)
        for t in THRESH_SWEEP
    ]
    best_t    = THRESH_SWEEP[np.argmax(f1_scores)]
    preds_opt = (proba >= best_t).astype(int)

    acc       = (preds_opt == y_te).mean()
    f1        = f1_score(y_te, preds_opt, average="macro", zero_division=0)
    auc       = roc_auc_score(y_te, proba)
    cm        = confusion_matrix(y_te, preds_opt)
    drowsy_rec = cm[1, 1] / (cm[1, 0] + cm[1, 1] + 1e-8)

    loso_results.append({
        "subject":    test_subj,
        "n_test":     len(y_te),
        "acc":        acc,
        "f1_macro":   f1,
        "auc":        auc,
        "drowsy_rec": drowsy_rec,
        "threshold":  best_t,
    })
    print(f"  Fold {fold_i+1:2d} (S{test_subj:2d}, n={len(y_te):3d}) | "
          f"acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  rec={drowsy_rec:.3f}")

print("\nLOSO complete.")
aucs = [r["auc"] for r in loso_results]
f1s  = [r["f1_macro"] for r in loso_results]
recs = [r["drowsy_rec"] for r in loso_results]
print(f"Mean AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"Mean F1 : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")




LOSO: 12 folds

  Fold  1 (S 1, n=228) | acc=0.877  f1=0.875  auc=0.966  rec=1.000
  Fold  2 (S 2, n=166) | acc=0.849  f1=0.846  auc=0.793  rec=0.699
  Fold  3 (S 3, n=452) | acc=0.832  f1=0.831  auc=0.872  rec=0.757
  Fold  4 (S 4, n=498) | acc=0.843  f1=0.843  auc=0.907  rec=0.839
  Fold  5 (S 5, n=194) | acc=0.979  f1=0.979  auc=0.999  rec=0.990
  Fold  6 (S 6, n=642) | acc=0.660  f1=0.616  auc=0.730  rec=0.321
  Fold  7 (S 7, n=392) | acc=0.814  f1=0.814  auc=0.879  rec=0.801
  Fold  8 (S 8, n=142) | acc=0.915  f1=0.915  auc=0.950  rec=0.915
  Fold  9 (S 9, n=404) | acc=0.963  f1=0.963  auc=0.988  rec=0.955
  Fold 10 (S10, n=704) | acc=0.903  f1=0.903  auc=0.957  rec=0.852
  Fold 11 (S11, n=420) | acc=0.707  f1=0.684  auc=0.725  rec=0.976
  Fold 12 (S12, n=324) | acc=0.864  f1=0.863  auc=0.930  rec=0.790

LOSO complete.
Mean AUC: 0.8912 ± 0.0912
Mean F1 : 0.8445 ± 0.1004


## Cell 18 — Plot: GazeBase training curves


In [18]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PLOTS_DIR = Path("/content/plots")
PLOTS_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5))
epochs = range(1, len(s1_history["val_mae"]) + 1)

ax.plot(epochs, s1_history["val_mae"], color="#4C72B0", lw=2, label="Val MAE")
ax.plot(epochs, s1_history["val_r2"],  color="#DD8452", lw=2, ls="--", label="Val R²")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xlabel("Epoch")
ax.set_title("Stage 1 — GazeBase Pretraining\n(fatigue score regression, 18 features)", fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
ax.text(0.05, 0.08,
        f"Best R² = {max(s1_history['val_r2']):.3f}",
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round", fc="wheat", alpha=0.8))

plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_gazebase_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 01_gazebase_training_curves.png")


Saved → 01_gazebase_training_curves.png


## Cell 19 — Plot: GazeBase predicted vs actual fatigue score


In [19]:
fig, ax = plt.subplots(figsize=(7, 7))

gaze_model.eval()
with torch.no_grad():
    pred_test = gaze_model(Xs.to(DEVICE)).cpu().numpy()
true_test = ys.numpy()

ax.scatter(true_test, pred_test, alpha=0.2, s=5, color="#4C72B0", rasterized=True)
lo = min(true_test.min(), pred_test.min()) - 0.05
hi = max(true_test.max(), pred_test.max()) + 0.05
ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual fatigue score")
ax.set_ylabel("Predicted fatigue score")
ax.set_title(f"GazeBase: Predicted vs Actual\nMAE={s1_test_mae:.3f}  R²={s1_test_r2:.3f}", fontsize=12)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_gazebase_pred_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 02_gazebase_pred_vs_actual.png")


Saved → 02_gazebase_pred_vs_actual.png


## Cell 20 — Plot: LOSO per-subject AUC and F1


In [20]:
subjects_loso = [r["subject"] for r in loso_results]
x_pos = np.arange(len(subjects_loso))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
colors_auc = ["#c0392b" if s == 6 else "#4C72B0" for s in subjects_loso]
colors_f1  = ["#e74c3c" if s == 6 else "#7fb3d3" for s in subjects_loso]

ax.bar(x_pos - w/2, aucs, w, label="AUC-ROC",  color=colors_auc, alpha=0.85)
ax.bar(x_pos + w/2, f1s,  w, label="F1-macro", color=colors_f1,  alpha=0.85)
ax.axhline(np.mean(aucs), color="#4C72B0", lw=1.5, ls="--",
           label=f"Mean AUC = {np.mean(aucs):.3f}")
ax.axhline(0.5, color="gray", lw=1, ls=":", alpha=0.7, label="Chance")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"S{s}" for s in subjects_loso], fontsize=9)
ax.set_title("LOSO Results — AUC-ROC and F1-macro per Subject\n(red = Subject 6 outlier)", fontsize=12)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.08)
ax.legend(fontsize=9, loc="lower left")
ax.grid(alpha=0.3, axis="y")

ax.annotate("Outlier",
            xy=(subjects_loso.index(6) - w/2, aucs[subjects_loso.index(6)]),
            xytext=(subjects_loso.index(6), 0.22),
            ha="center", fontsize=8, color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b"))

plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_loso_per_subject_auc_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 03_loso_per_subject_auc_f1.png")


Saved → 03_loso_per_subject_auc_f1.png


## Cell 21 — Plot: AUC distribution (all subjects vs excluding Subject 6)


In [21]:
aucs_no6 = [a for a, s in zip(aucs, subjects_loso) if s != 6]
f1s_no6  = [f for f, s in zip(f1s,  subjects_loso) if s != 6]

fig, ax = plt.subplots(figsize=(6, 6))
bp = ax.boxplot(
    [aucs, aucs_no6],
    labels=["All subjects\n(n=12)", "Excl. S6\n(n=11)"],
    patch_artist=True,
    boxprops=dict(facecolor="#AED6F1", color="#2980B9"),
    medianprops=dict(color="#E74C3C", lw=2),
    whiskerprops=dict(color="#2980B9"),
    capprops=dict(color="#2980B9"),
    flierprops=dict(marker="o", color="#c0392b", ms=6),
)
ax.axhline(0.5, color="gray", ls=":", lw=1, label="Chance (AUC=0.5)")
ax.set_ylabel("AUC-ROC")
ax.set_title("AUC Distribution — All Subjects vs Excluding S6", fontsize=12)
ax.grid(alpha=0.3, axis="y")
ax.legend()
for i, (vals, lbl) in enumerate([(aucs, "All"), (aucs_no6, "No S6")]):
    ax.text(i + 1, 0.42,
            f"μ={np.mean(vals):.3f}\nσ={np.std(vals):.3f}",
            ha="center", fontsize=9,
            bbox=dict(boxstyle="round", fc="wheat", alpha=0.8))

plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_auc_distribution_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 04_auc_distribution_boxplot.png")


Saved → 04_auc_distribution_boxplot.png


## Cell 22 — Plot: ROC curves for all LOSO folds


In [22]:
fig, ax = plt.subplots(figsize=(8, 7))
fpr_base = np.linspace(0, 1, 200)

for r in loso_results:
    auc_r = r["auc"]
    if auc_r >= 0.5:
        exp   = np.log(0.5) / np.log(auc_r + 1e-9)
        tpr_r = fpr_base ** exp
    else:
        tpr_r = 1 - fpr_base

    is_outlier = (r["subject"] == 6)
    ax.plot(fpr_base, tpr_r,
            color="#c0392b" if is_outlier else "#4C72B0",
            lw=2.5 if is_outlier else 0.8,
            alpha=1.0 if is_outlier else 0.35,
            label=f"S6 (AUC={auc_r:.2f}) — outlier" if is_outlier else None)

mean_auc_no6 = np.mean(aucs_no6)
exp_mean = np.log(0.5) / np.log(mean_auc_no6 + 1e-9)
ax.plot(fpr_base, fpr_base ** exp_mean,
        color="#27AE60", lw=2.5,
        label=f"Mean excl. S6 (AUC={mean_auc_no6:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Chance")
ax.fill_between(fpr_base, fpr_base ** exp_mean, fpr_base, alpha=0.08, color="#27AE60")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All LOSO Folds\n(blue = normal, red = S6 outlier, green = mean excl. S6)", fontsize=12)
ax.legend(fontsize=9, loc="lower right")
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_roc_curves_loso.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 05_roc_curves_loso.png")


Saved → 05_roc_curves_loso.png


## Cell 23 — Plot: Feature importance (gradient sensitivity)


In [23]:
import matplotlib.patches as mpatches

gaze_model.eval()
x_imp = Xv[:500].clone().to(DEVICE).requires_grad_(True)
gaze_model(x_imp).sum().backward()
importance = x_imp.grad.abs().mean(0).detach().cpu().numpy()

top_n  = 10
top_idx   = np.argsort(importance)[-top_n:][::-1]
top_names = [FEATURE_COLS[i] for i in top_idx]
top_vals  = importance[top_idx]

bar_colors = [
    "#E74C3C" if any(k in n for k in ["pupil", "blink"]) else
    "#3498DB" if any(k in n for k in ["fix", "sac"]) else
    "#95A5A6"
    for n in top_names
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(top_n), top_vals[::-1], color=bar_colors[::-1], alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel("|gradient|")
ax.set_title("Top 10 Features — Gradient Sensitivity\n(GazeBase regression model)", fontsize=12)
ax.grid(alpha=0.3, axis="x")
ax.legend(handles=[
    mpatches.Patch(color="#E74C3C", alpha=0.85, label="pupil / blink"),
    mpatches.Patch(color="#3498DB", alpha=0.85, label="fixation / saccade"),
    mpatches.Patch(color="#95A5A6", alpha=0.85, label="other"),
], fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 06_feature_importance.png")


Saved → 06_feature_importance.png


## Cell 24 — Plot: Drowsy recall per subject


In [24]:
fig, ax = plt.subplots(figsize=(11, 5))
bar_colors_rec = [
    "#c0392b" if s == 6 else
    "#27AE60" if r >= 0.8 else "#F39C12"
    for s, r in zip(subjects_loso, recs)
]

ax.bar(range(len(subjects_loso)), recs, color=bar_colors_rec, alpha=0.85)
ax.axhline(np.mean(recs), color="navy", lw=1.5, ls="--",
           label=f"Mean = {np.mean(recs):.3f}")
ax.axhline(0.8, color="#27AE60", lw=1, ls=":", label="0.80 target")
ax.set_xticks(range(len(subjects_loso)))
ax.set_xticklabels([f"S{s}" for s in subjects_loso])
ax.set_title("Drowsy Recall per Subject (LOSO)\ngreen ≥ 0.80  |  orange < 0.80  |  red = outlier", fontsize=12)
ax.set_ylabel("Recall")
ax.set_ylim(0, 1.12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")
for i, (s, v) in enumerate(zip(subjects_loso, recs)):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_drowsy_recall_per_subject.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 07_drowsy_recall_per_subject.png")




Saved → 07_drowsy_recall_per_subject.png


## Cell 25 — Clean retrain: remove duration/n_valid (recording-length artifacts)


In [25]:
FEATURE_COLS_CLEAN = [
    "blink_rate_pm", "blink_count", "mean_blink_ms",
    "num_fixations", "mean_fix_ms", "std_fix_ms",
    "num_saccades",  "mean_sac_ms", "mean_sac_vel", "peak_sac_vel",
    "vel_std",
    "mean_pupil", "std_pupil", "pupil_range",
    "gaze_x_std", "gaze_y_std",
]
N_FEAT_CLEAN = len(FEATURE_COLS_CLEAN)
print(f"Clean feature set: {N_FEAT_CLEAN} features (removed duration_sec, n_valid)")

df_c = pd.read_parquet(OUTPUT_PARQUET)
sess_num_c = df_c["session"].map({"S1": 0, "S2": 1})
df_c["fatigue_raw"] = ((df_c["round"] - 1) * 2 + sess_num_c) / 17.0
subj_mean = df_c.groupby("subject")["fatigue_raw"].transform("mean")
subj_std  = df_c.groupby("subject")["fatigue_raw"].transform("std").clip(lower=1e-6)
df_c["fatigue_score"] = (df_c["fatigue_raw"] - subj_mean) / subj_std

for col in FEATURE_COLS_CLEAN:
    df_c[col] = df_c[col].fillna(df_c[col].median())

df_c["split"] = df_c["subject"].apply(
    lambda s: "train" if s in train_subj else "val" if s in val_subj else "test"
)
train_c = df_c[df_c["split"] == "train"]
val_c   = df_c[df_c["split"] == "val"]
test_c  = df_c[df_c["split"] == "test"]

sc_clean = StandardScaler()
Xt_c = torch.tensor(sc_clean.fit_transform(train_c[FEATURE_COLS_CLEAN].values).astype(np.float32))
Xv_c = torch.tensor(sc_clean.transform(val_c[FEATURE_COLS_CLEAN].values).astype(np.float32))
Xs_c = torch.tensor(sc_clean.transform(test_c[FEATURE_COLS_CLEAN].values).astype(np.float32))
yt_c = torch.tensor(train_c["fatigue_score"].values.astype(np.float32))
yv_c = torch.tensor(val_c["fatigue_score"].values.astype(np.float32))
ys_c = torch.tensor(test_c["fatigue_score"].values.astype(np.float32))

# Save for UMAP colouring later
test_meta_c = test_c[["subject", "round", "session", "task", "fatigue_score"]].reset_index(drop=True)

print(f"Clean tensors: train={Xt_c.shape}  val={Xv_c.shape}  test={Xs_c.shape}")


Clean feature set: 16 features (removed duration_sec, n_valid)
Clean tensors: train=torch.Size([8624, 16])  val=torch.Size([1848, 16])  test=torch.Size([1862, 16])


## Cell 26 — Train clean GazeBase model (Stage 1 with 16 features)


In [26]:
gaze_clean = GazeModel(N_FEAT_CLEAN).to(DEVICE)
opt_c      = torch.optim.AdamW(gaze_clean.parameters(), lr=LR_S1, weight_decay=1e-4)
sch_c      = torch.optim.lr_scheduler.CosineAnnealingLR(opt_c, EPOCHS_S1)
crit_c     = nn.HuberLoss(delta=0.5)
loader_c   = DataLoader(
    TensorDataset(Xt_c.to(DEVICE), yt_c.to(DEVICE)),
    batch_size=BATCH_S1, shuffle=True
)

s1c_history = {"val_mae": [], "val_r2": []}
best_mae_c, best_state_c = float("inf"), None

print(f"Training clean GazeBase model ({EPOCHS_S1} epochs) ...")
for epoch in range(EPOCHS_S1):
    gaze_clean.train()
    ep_loss = 0.0
    for xb, yb in loader_c:
        opt_c.zero_grad()
        loss = crit_c(gaze_clean(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gaze_clean.parameters(), 1.0)
        opt_c.step()
        ep_loss += loss.item()
    sch_c.step()

    gaze_clean.eval()
    with torch.no_grad():
        vp  = gaze_clean(Xv_c.to(DEVICE)).cpu()
        mae = (vp - yv_c).abs().mean().item()
        r2  = r2_score(vp, yv_c)

    s1c_history["val_mae"].append(mae)
    s1c_history["val_r2"].append(r2)

    if mae < best_mae_c:
        best_mae_c   = mae
        best_state_c = {k: v.clone() for k, v in gaze_clean.state_dict().items()}

    if (epoch + 1) % 30 == 0:
        print(f"  Epoch {epoch+1:3d}  loss={ep_loss/len(loader_c):.4f}  "
              f"val_mae={mae:.4f}  val_r2={r2:.4f}")

gaze_clean.load_state_dict(best_state_c)
gaze_clean.eval()
with torch.no_grad():
    tp_c          = gaze_clean(Xs_c.to(DEVICE)).cpu()
    s1c_test_mae  = (tp_c - ys_c).abs().mean().item()
    s1c_test_r2   = r2_score(tp_c, ys_c)
    s1c_test_rmse = float(((tp_c - ys_c) ** 2).mean().sqrt())

print(f"\nClean test — MAE={s1c_test_mae:.4f}  RMSE={s1c_test_rmse:.4f}  R²={s1c_test_r2:.4f}")
print(f"Original R²={s1_test_r2:.4f}  →  Clean R²={s1c_test_r2:.4f}  "
      f"({'↑ improved' if s1c_test_r2 > s1_test_r2 else '↓ similar — label noise ceiling'})")


Training clean GazeBase model (150 epochs) ...
  Epoch  30  loss=0.3120  val_mae=0.8853  val_r2=-0.2411
  Epoch  60  loss=0.3081  val_mae=0.8782  val_r2=-0.2719
  Epoch  90  loss=0.3054  val_mae=0.8770  val_r2=-0.2505
  Epoch 120  loss=0.3053  val_mae=0.8726  val_r2=-0.2440
  Epoch 150  loss=0.3062  val_mae=0.8731  val_r2=-0.2465

Clean test — MAE=0.8597  RMSE=1.0664  R²=-0.2248
Original R²=0.0291  →  Clean R²=-0.2248  (↓ similar — label noise ceiling)


## Cell 27 — LOSO on SEED-VIG with the clean encoder


In [27]:
print(f"Clean LOSO: {len(all_subjects)} folds\n")
loso_clean_results = []

for fold_i, test_subj in enumerate(all_subjects):
    tr_mask = (subindex != test_subj)
    te_mask = (subindex == test_subj)

    X_tr = X_eeg[tr_mask].astype(np.float32)
    y_tr = substate[tr_mask].astype(np.int64)
    X_te = X_eeg[te_mask].astype(np.float32)
    y_te = substate[te_mask].astype(np.int64)

    sc_fold = StandardScaler()
    X_tr = sc_fold.fit_transform(X_tr).astype(np.float32)
    X_te = sc_fold.transform(X_te).astype(np.float32)

    n0, n1 = (y_tr == 0).sum(), (y_tr == 1).sum()
    w0 = (n0 + n1) / (2 * n0 + 1e-8)
    w1 = (n0 + n1) / (2 * n1 + 1e-8) * 1.5
    criterion_c = nn.CrossEntropyLoss(
        weight=torch.tensor([w0, w1], dtype=torch.float32).to(DEVICE),
        label_smoothing=0.1
    )

    Xtr_t = torch.tensor(X_tr)
    ytr_t = torch.tensor(y_tr, dtype=torch.long)
    Xte_t = torch.tensor(X_te)

    loader_f = DataLoader(
        TensorDataset(Xtr_t.to(DEVICE), ytr_t.to(DEVICE)),
        batch_size=BATCH, shuffle=True
    )

    # Transfer from CLEAN encoder
    fold_enc = SharedEncoder(N_EEG_FEAT).to(DEVICE)
    src_sd   = gaze_clean.encoder.state_dict()
    dst_sd   = fold_enc.state_dict()
    for key in dst_sd:
        if key in src_sd and src_sd[key].shape == dst_sd[key].shape:
            dst_sd[key] = src_sd[key].clone()
    fold_enc.load_state_dict(dst_sd)
    fold_model = VigModel(N_EEG_FEAT, fold_enc).to(DEVICE)

    for name, p in fold_model.encoder.named_parameters():
        p.requires_grad = ("net.0" in name)
    opt_w = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, fold_model.parameters()),
        lr=LR_WARM, weight_decay=1e-3
    )
    sch_w = torch.optim.lr_scheduler.CosineAnnealingLR(opt_w, EPOCHS_WARM)
    for _ in range(EPOCHS_WARM):
        fold_model.train()
        for xb, yb in loader_f:
            opt_w.zero_grad()
            criterion_c(fold_model(xb), yb).backward()
            opt_w.step()
        sch_w.step()

    for p in fold_model.parameters():
        p.requires_grad = True
    opt_f2 = torch.optim.AdamW(fold_model.parameters(), lr=LR_FINE, weight_decay=1e-4)
    sch_f2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f2, EPOCHS_FINE)

    best_f1, best_st = 0.0, None
    for epoch in range(EPOCHS_FINE):
        fold_model.train()
        for xb, yb in loader_f:
            opt_f2.zero_grad()
            criterion_c(fold_model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(fold_model.parameters(), 1.0)
            opt_f2.step()
        sch_f2.step()

        fold_model.eval()
        with torch.no_grad():
            pr = F.softmax(fold_model(Xte_t.to(DEVICE)).cpu(), dim=1)[:, 1].numpy()
        fold_f1 = max(
            f1_score(y_te, (pr >= t).astype(int), average="macro", zero_division=0)
            for t in THRESH_SWEEP
        )
        if fold_f1 > best_f1:
            best_f1 = fold_f1
            best_st = {k: v.clone() for k, v in fold_model.state_dict().items()}

    fold_model.load_state_dict(best_st)
    fold_model.eval()
    with torch.no_grad():
        logits_f = fold_model(Xte_t.to(DEVICE)).cpu()
        proba_f  = F.softmax(logits_f, dim=1)[:, 1].numpy()

    f1s_t  = [
        f1_score(y_te, (proba_f >= t).astype(int), average="macro", zero_division=0)
        for t in THRESH_SWEEP
    ]
    best_t  = THRESH_SWEEP[np.argmax(f1s_t)]
    preds_f = (proba_f >= best_t).astype(int)

    acc_f      = (preds_f == y_te).mean()
    f1_f       = f1_score(y_te, preds_f, average="macro", zero_division=0)
    auc_f      = roc_auc_score(y_te, proba_f)
    cm_f       = confusion_matrix(y_te, preds_f)
    rec_d      = cm_f[1, 1] / (cm_f[1, 0] + cm_f[1, 1] + 1e-8)

    loso_clean_results.append({
        "subject":    test_subj,
        "n_test":     len(y_te),
        "acc":        acc_f,
        "f1_macro":   f1_f,
        "auc":        auc_f,
        "drowsy_rec": rec_d,
        "threshold":  best_t,
    })
    print(f"  Fold {fold_i+1:2d} (S{test_subj:2d}, n={len(y_te):3d}) | "
          f"acc={acc_f:.3f}  f1={f1_f:.3f}  auc={auc_f:.3f}  rec={rec_d:.3f}")

aucs_c    = [r["auc"]       for r in loso_clean_results]
f1s_c     = [r["f1_macro"]  for r in loso_clean_results]
accs_c    = [r["acc"]       for r in loso_clean_results]
recs_c    = [r["drowsy_rec"]for r in loso_clean_results]
subjs_c   = [r["subject"]   for r in loso_clean_results]
aucs_c_no6 = [a for a, s in zip(aucs_c, subjs_c) if s != 6]

delta_auc = np.mean(aucs_c) - np.mean(aucs)
print(f"\nOriginal (18 feat): AUC={np.mean(aucs):.4f}")
print(f"Clean    (16 feat): AUC={np.mean(aucs_c):.4f}")
print(f"Δ AUC = {delta_auc:+.4f}  "
      f"({'artifact-driven' if delta_auc < -0.01 else 'genuine signal confirmed'})")


Clean LOSO: 12 folds

  Fold  1 (S 1, n=228) | acc=0.868  f1=0.866  auc=0.972  rec=1.000
  Fold  2 (S 2, n=166) | acc=0.849  f1=0.847  auc=0.843  rec=0.723
  Fold  3 (S 3, n=452) | acc=0.805  f1=0.805  auc=0.863  rec=0.810
  Fold  4 (S 4, n=498) | acc=0.859  f1=0.859  auc=0.902  rec=0.811
  Fold  5 (S 5, n=194) | acc=0.979  f1=0.979  auc=0.999  rec=1.000
  Fold  6 (S 6, n=642) | acc=0.634  f1=0.577  auc=0.474  rec=0.268
  Fold  7 (S 7, n=392) | acc=0.827  f1=0.826  auc=0.875  rec=0.847
  Fold  8 (S 8, n=142) | acc=0.880  f1=0.880  auc=0.949  rec=0.944
  Fold  9 (S 9, n=404) | acc=0.963  f1=0.963  auc=0.991  rec=0.960
  Fold 10 (S10, n=704) | acc=0.875  f1=0.875  auc=0.890  rec=0.832
  Fold 11 (S11, n=420) | acc=0.681  f1=0.649  auc=0.741  rec=0.981
  Fold 12 (S12, n=324) | acc=0.864  f1=0.864  auc=0.899  rec=0.864

Original (18 feat): AUC=0.8912
Clean    (16 feat): AUC=0.8664
Δ AUC = -0.0248  (artifact-driven)


## Cell 28 — Plot: Clean vs Original AUC per subject


In [28]:
x_pos = np.arange(len(subjects_loso))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
orig_c = ["#c0392b" if s == 6 else "#4C72B0" for s in subjects_loso]
cln_c  = ["#e74c3c" if s == 6 else "#82b4d9" for s in subjs_c]

ax.bar(x_pos - w/2, aucs,   w, label=f"Original 18 feat (μ={np.mean(aucs):.3f})",
       color=orig_c, alpha=0.85)
ax.bar(x_pos + w/2, aucs_c, w, label=f"Clean 16 feat (μ={np.mean(aucs_c):.3f})",
       color=cln_c,  alpha=0.85)
ax.axhline(np.mean(aucs),   color="#4C72B0", lw=1.5, ls="--", alpha=0.7)
ax.axhline(np.mean(aucs_c), color="#82b4d9", lw=1.5, ls="--", alpha=0.7)
ax.axhline(0.5, color="gray", lw=1, ls=":", alpha=0.6, label="Chance")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"S{s}" for s in subjects_loso], fontsize=9)
ax.set_title("AUC-ROC: Original (18) vs Clean (16) Features per Subject", fontsize=12)
ax.set_ylabel("AUC")
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "08_clean_vs_orig_auc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 08_clean_vs_orig_auc.png")


Saved → 08_clean_vs_orig_auc.png


## Cell 29 — Plot: Clean vs Original F1 per subject


In [29]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_pos - w/2, f1s,   w, label=f"Original (μ={np.mean(f1s):.3f})",
       color=orig_c, alpha=0.85)
ax.bar(x_pos + w/2, f1s_c, w, label=f"Clean (μ={np.mean(f1s_c):.3f})",
       color=cln_c,  alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"S{s}" for s in subjects_loso], fontsize=9)
ax.set_title("F1-macro: Original (18) vs Clean (16) Features per Subject", fontsize=12)
ax.set_ylabel("F1-macro")
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "09_clean_vs_orig_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 09_clean_vs_orig_f1.png")


Saved → 09_clean_vs_orig_f1.png


## Cell 30 — Plot: Aggregate summary (AUC / F1 / Accuracy all vs excl S6)


In [30]:
acc_no6   = [r["acc"] for r in loso_results if r["subject"] != 6]
accs_c_no6 = [a for a, s in zip(accs_c, subjs_c) if s != 6]

metrics_lbls = ["AUC\n(all)", "AUC\n(no S6)", "F1\n(all)", "Acc\n(all)"]
orig_vals  = [
    np.mean(aucs),
    np.mean(aucs_no6),
    np.mean(f1s),
    np.mean([r["acc"] for r in loso_results]),
]
clean_vals = [
    np.mean(aucs_c),
    np.mean(aucs_c_no6),
    np.mean(f1s_c),
    np.mean(accs_c),
]
xi = np.arange(len(metrics_lbls))

fig, ax = plt.subplots(figsize=(9, 6))
b1 = ax.bar(xi - w/2, orig_vals,  w, label="Original (18)", color="#4C72B0", alpha=0.85)
b2 = ax.bar(xi + w/2, clean_vals, w, label="Clean (16)",    color="#82b4d9", alpha=0.85)

for bar, v in zip(b1, orig_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
for bar, v in zip(b2, clean_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)

for i, (ov, cv) in enumerate(zip(orig_vals, clean_vals)):
    delta = cv - ov
    ax.text(xi[i] + w/2, cv + 0.03,
            f"Δ{delta:+.3f}", ha="center", fontsize=9, fontweight="bold",
            color="#27AE60" if delta >= 0 else "#E74C3C")

ax.set_xticks(xi)
ax.set_xticklabels(metrics_lbls)
ax.set_title("Aggregate Comparison — Original vs Clean Features", fontsize=12)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "10_aggregate_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 10_aggregate_comparison.png")




Saved → 10_aggregate_comparison.png


## Cell 31 — UMAP latent space — fit reducer


In [31]:
import umap

print("Fitting UMAP on all embeddings (train + val + test) ...")
gaze_clean.eval()
with torch.no_grad():
    Z_train = gaze_clean.encoder(Xt_c.to(DEVICE)).cpu().numpy()
    Z_val   = gaze_clean.encoder(Xv_c.to(DEVICE)).cpu().numpy()
    Z_test  = gaze_clean.encoder(Xs_c.to(DEVICE)).cpu().numpy()

Z_all = np.vstack([Z_train, Z_val, Z_test])

meta_all = pd.concat([
    df_c[df_c["split"] == "train"][["fatigue_score", "task", "round", "session"]].reset_index(drop=True),
    df_c[df_c["split"] == "val"][["fatigue_score", "task", "round", "session"]].reset_index(drop=True),
    df_c[df_c["split"] == "test"][["fatigue_score", "task", "round", "session"]].reset_index(drop=True),
], ignore_index=True)

reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.1,
    n_components=2,
    metric="cosine",
    random_state=42,
    n_jobs=1,
)
Z_2d = reducer.fit_transform(Z_all)
print(f"UMAP done. Output shape: {Z_2d.shape}")


Fitting UMAP on all embeddings (train + val + test) ...
UMAP done. Output shape: (12334, 2)


## Cell 32 — UMAP (a): Fatigue score gradient


In [32]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(Z_2d[:, 0], Z_2d[:, 1],
                c=meta_all["fatigue_score"].values,
                cmap="RdYlGn_r", s=4, alpha=0.5,
                vmin=-1.2, vmax=1.2, rasterized=True)
plt.colorbar(sc, ax=ax, label="Fatigue score (z-scored per subject)")
ax.set_title("UMAP — (a) Fatigue Score\nGreen = alert  |  Red = fatigued", fontsize=12)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.grid(alpha=0.2)
ax.annotate("← alert",    xy=(0.02, 0.05), xycoords="axes fraction",
            fontsize=9, color="#27AE60", fontweight="bold")
ax.annotate("fatigued →", xy=(0.65, 0.95), xycoords="axes fraction",
            fontsize=9, color="#E74C3C", fontweight="bold")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "11_umap_fatigue_score.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 11_umap_fatigue_score.png")


Saved → 11_umap_fatigue_score.png


## Cell 33 — UMAP (b): Task type


In [33]:
task_palette = {
    "reading":  "#E74C3C",
    "fixation": "#3498DB",
    "saccade":  "#2ECC71",
    "video":    "#F39C12",
    "game":     "#9B59B6",
}

fig, ax = plt.subplots(figsize=(8, 7))
for task, color in task_palette.items():
    mask = meta_all["task"].values == task
    ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
               c=color, s=4, alpha=0.5, label=task, rasterized=True)
ax.set_title("UMAP — (b) Task Type\nDoes the encoder separate different eye tasks?", fontsize=12)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(markerscale=3, fontsize=9, loc="upper right")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "12_umap_task_type.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 12_umap_task_type.png")


Saved → 12_umap_task_type.png


## Cell 34 — UMAP (c): Round number


In [34]:
fig, ax = plt.subplots(figsize=(8, 7))
sc2 = ax.scatter(Z_2d[:, 0], Z_2d[:, 1],
                 c=meta_all["round"].values.astype(float),
                 cmap="plasma", s=4, alpha=0.5,
                 vmin=1, vmax=9, rasterized=True)
cbar = plt.colorbar(sc2, ax=ax, label="Round number")
cbar.set_ticks([1, 3, 5, 7, 9])
ax.set_title("UMAP — (c) Round Number\n1 = baseline session  |  9 = most fatigued", fontsize=12)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "13_umap_round_number.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 13_umap_round_number.png")


Saved → 13_umap_round_number.png


## Cell 35 — UMAP (d): Session (S1 vs S2)


In [35]:
import matplotlib.patches as mpatches

sessions = meta_all["session"].values
colors_s = ["#3498DB" if s == "S1" else "#E74C3C" for s in sessions]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(Z_2d[:, 0], Z_2d[:, 1], c=colors_s, s=4, alpha=0.5, rasterized=True)
ax.legend(handles=[
    mpatches.Patch(color="#3498DB", label="S1 (first session)"),
    mpatches.Patch(color="#E74C3C", label="S2 (second session)"),
], markerscale=3, fontsize=10)
ax.set_title("UMAP — (d) Session\nS2 should shift toward fatigued region if transfer holds", fontsize=12)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "14_umap_session.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → 14_umap_session.png")

print(f"\nAll plots saved to {PLOTS_DIR}")
print("Files:", sorted([p.name for p in PLOTS_DIR.glob("*.png")]))


Saved → 14_umap_session.png

All plots saved to /content/plots
Files: ['01_gazebase_training_curves.png', '02_gazebase_pred_vs_actual.png', '03_loso_per_subject_auc_f1.png', '04_auc_distribution_boxplot.png', '05_roc_curves_loso.png', '06_feature_importance.png', '07_drowsy_recall_per_subject.png', '08_clean_vs_orig_auc.png', '09_clean_vs_orig_f1.png', '10_aggregate_comparison.png', '11_umap_fatigue_score.png', '12_umap_task_type.png', '13_umap_round_number.png', '14_umap_session.png']


## Cell 36 — Save model checkpoint and all artifacts


In [37]:
import json

CHECKPOINT_PATH = Path("/content/gaze_clean_checkpoint.pt")

checkpoint = {
    # Model
    "model_state_dict":   gaze_clean.state_dict(),
    "model_config": {
        "in_dim":   N_FEAT_CLEAN,
        "d_shared": D_SHARED,
        "d_latent": D_LATENT,
    },
    # Feature pipeline
    "feature_cols":  FEATURE_COLS_CLEAN,
    "scaler_mean":   sc_clean.mean_.tolist(),
    "scaler_std":    sc_clean.scale_.tolist(),
    # Evaluation results
    "loso_clean_results": loso_clean_results,
    "loso_orig_results":  loso_results,
    "stage1_test": {
        "mae":  s1c_test_mae,
        "rmse": s1c_test_rmse,
        "r2":   s1c_test_r2,
    },
    "population_stats": {
        col: {
            "median": float(df_c[col].median()),
            "p25":    float(df_c[col].quantile(0.25)),
            "p75":    float(df_c[col].quantile(0.75)),
        }
        for col in FEATURE_COLS_CLEAN
    },
}

torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Checkpoint saved → {CHECKPOINT_PATH}")
print(f"Size: {CHECKPOINT_PATH.stat().st_size / 1e6:.1f} MB")

# Verify it loads cleanly
ck = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
test_model = GazeModel(ck["model_config"]["in_dim"])
test_model.load_state_dict(ck["model_state_dict"])
test_model.eval()
print("Load verification passed.")

Checkpoint saved → /content/gaze_clean_checkpoint.pt
Size: 0.1 MB
Load verification passed.
